# Push to Chinook to Hugging face

We'll convert the tables in `chinook` to parquet files in a directory and then push the directory to Hugging Face.


## Chinook data set

See the lecture on SQLite3 using the Chinook data set to set up the software, database, and tables, as well as for the links to ancillary information about the data set.


In [1]:
import sqlite3 as db
import pandas as pd
import contextlib


In [2]:
%%bash
curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip


In [3]:
!unzip -u chinook.zip


Archive:  chinook.zip
  inflating: chinook.db              


In [4]:
!ls -l


total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Jul  9 06:06 chinook.zip
drwxr-xr-x 1 root root   4096 Jun  4 13:32 sample_data


Verify sqlite works.

In [5]:
query = '''
  SELECT
    name
  FROM
    sqlite_master
  WHERE
    type='table' AND
    name not like "sqlite_%"
;
'''
print(query)


  SELECT
    name
  FROM
    sqlite_master
  WHERE
    type='table' AND
    name not like "sqlite_%"
;



In [6]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  tables = pd.read_sql_query( query , db_con)

tables


,name
0,albums
1,artists
2,customers
3,employees
4,genres
5,invoices
6,invoice_items
7,media_types
8,playlists
9,playlist_track


## Read each table and convert to parquet


In [16]:
# create target folder
!mkdir -p chinook


In [17]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  for table in tables["name"]:
    print(table)
    query = f'''
      select * from {table} ;
    '''
    df = pd.read_sql_query( query, db_con)
    display(df.shape)
    output_path = f'./chinook/{table}.parquet'
    df.to_parquet(output_path, index=False)


albums


(347, 3)

artists


(275, 2)

customers


...

(18, 2)

playlist_track


(8715, 2)

tracks


(3503, 9)

In [18]:
# verify
!ls -l chinook


total 288
-rw-r--r-- 1 root root  12279 Jul  9 06:15 albums.parquet
-rw-r--r-- 1 root root   8433 Jul  9 06:15 artists.parquet
-rw-r--r-- 1 root root  14160 Jul  9 06:15 customers.parquet
-rw-r--r-- 1 root root   9458 Jul  9 06:15 employees.parquet
-rw-r--r-- 1 root root   2088 Jul  9 06:15 genres.parquet
-rw-r--r-- 1 root root  30063 Jul  9 06:15 invoice_items.parquet
-rw-r--r-- 1 root root  14951 Jul  9 06:15 invoices.parquet
-rw-r--r-- 1 root root   1832 Jul  9 06:15 media_types.parquet
-rw-r--r-- 1 root root   1976 Jul  9 06:15 playlists.parquet
-rw-r--r-- 1 root root  29024 Jul  9 06:15 playlist_track.parquet
-rw-r--r-- 1 root root 143746 Jul  9 06:15 tracks.parquet


## Read from parquet



In [21]:
files = !ls -1 chinook/*.parquet
files


['chinook/albums.parquet',
 'chinook/artists.parquet',
 'chinook/customers.parquet',
 'chinook/employees.parquet',
 'chinook/genres.parquet',
 'chinook/invoice_items.parquet',
 'chinook/invoices.parquet',
 'chinook/media_types.parquet',
 'chinook/playlists.parquet',
 'chinook/playlist_track.parquet',
 'chinook/tracks.parquet']

In [23]:
for table in files:
  print(table)
  df = pd.read_parquet( table )
  display(df.shape)



chinook/albums.parquet


(347, 3)

chinook/artists.parquet


(275, 2)

chinook/customers.parquet


...

(18, 2)

chinook/playlist_track.parquet


(8715, 2)

chinook/tracks.parquet


(3503, 9)